In [ ]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans

import matplotlib.pyplot as plt

print("=== LIGHTWEIGHT CLUSTERING STARTED ===")

# Load ONLY required columns
usecols = ["shopno", "nooftrans", "noofrcs"]

chunks = pd.read_csv(
    "../data/processed/master_pds_data.csv",
    usecols=usecols,
    chunksize=80000
)

sample_data = []

for chunk in chunks:
    sample_data.append(chunk.sample(5000, random_state=42))

df = pd.concat(sample_data, ignore_index=True)

print("Loaded Shape:", df.shape)

# Feature engineering
df["utilization_ratio"] = df["nooftrans"] / (df["noofrcs"] + 1)

# Features
X = df[[
    "nooftrans",
    "noofrcs",
    "utilization_ratio"
]].fillna(0)

# Scaling
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# PCA
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

df["pca1"] = X_pca[:,0]
df["pca2"] = X_pca[:,1]

# KMeans
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
df["cluster"] = kmeans.fit_predict(X_scaled)

# Visualization
plt.figure(figsize=(8,5))

plt.scatter(
    df["pca1"],
    df["pca2"],
    c=df["cluster"],
    s=3
)

plt.title("PCA Cluster Visualization")
plt.show()

# Save output
df.to_csv("../data/processed/clustered_pds_data.csv", index=False)

print("Saved clustered_pds_data.csv")
print("=== CLUSTERING COMPLETED ===")

hellow
